# 3 · Fine-tuning with a verifier in the loop

Train a small open model to emit `Holding` JSON from a provision plus its
retrieved precedents, then run **expert iteration**: sample, verify with Lean,
keep only what verifies, retrain on that.

This is the standard recipe from neural theorem proving (DeepSeek-Prover,
Lean-STaR, and the expert-iteration line generally) applied to a domain where
the verifier checks something more interesting than type-correctness. Here a
sample survives only if its ratio is *legally* well-formed and it does not
contradict the precedents already formalised around it.

**Where to run this.** A T4 or better; Colab's free tier is enough for a 7B
model in 4-bit with LoRA. Everything before the training cell runs on CPU.

**Expected scale.** 43 seed pairs is far too few to fine-tune on. The realistic
sequence is: (1) bootstrap a few hundred more pairs with a strong API model
through notebook 2, keeping only verified ones; (2) fine-tune on those;
(3) iterate. Cell 6 below builds the dataset from whatever is on disk, so it
grows as you run notebook 2.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Locate the repository root whether this runs from notebooks/ or the root.
ROOT = Path.cwd()
while not (ROOT / "src" / "nomos").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("repo root:", ROOT)

# The built corpus is a reproducible artefact and is not shipped in the
# archive (it is 13 MB gzipped and rebuilds from data/raw in about a minute).
if not any((ROOT / "data/corpus").glob("provisions.jsonl*")):
    print("building the corpus from data/raw ...")
    subprocess.run([sys.executable, "-m", "nomos.build_corpus"], check=True,
                   env={**os.environ, "PYTHONPATH": str(ROOT / "src")})


## 1. Dependencies

Skip if already installed.

In [ ]:
# !pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.20" \
#                 "accelerate>=0.33" "bitsandbytes>=0.43" sentencepiece
import importlib
for m in ["torch", "transformers", "peft", "trl", "datasets"]:
    try:
        importlib.import_module(m)
        print(f"{m:14s} ok")
    except ImportError:
        print(f"{m:14s} MISSING -- uncomment the pip line above")

## 2. Build the training set

Each example is a chat exchange:

* **system** — the formalisation instructions from `nomos.prompts.SYSTEM`,
  including the three rules the verifier enforces;
* **user** — the factor glossary, the remedy grammar, the retrieved precedents
  with the reason each is shown, and the provision;
* **assistant** — the gold `Holding` as JSON.

The precedents in the user turn are retrieved *excluding the target itself*, so
the model never sees the answer it is being asked for.

In [ ]:
from nomos.schema import read_jsonl
from nomos.analogy import Holding, retrieve
from nomos import prompts

seed_rows = list(read_jsonl("data/seed/formalizations.jsonl"))
seed = [Holding(cite=r["cite"], situation=r["situation"], winner=r["winner"],
                remedy=r["remedy"], reason=r["reason"], tradition=r["tradition"],
                restatement=(r.get("text") or "")[:300]) for r in seed_rows]

def example_for(row, corpus):
    target = {"citation": row["cite"], "work": row["tradition"],
              "tradition": row["tradition"], "language": "en",
              "canonical": row["cite"], "text": row.get("text"),
              "source": row.get("source") or {}}
    user = prompts.build_prompt(target, [h for h in corpus if h.cite != row["cite"]],
                                guess_situation=row["situation"], k=6)
    answer = {"applicable": True,
              "restatement": (row.get("text") or "")[:200],
              "situation": row["situation"], "winner": row["winner"],
              "remedy": row["remedy"], "reason": row["reason"],
              "confidence": "high", "notes": ""}
    return {"messages": [
        {"role": "system", "content": prompts.SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False, indent=2)},
    ], "cite": row["cite"], "tradition": row["tradition"]}

examples = [example_for(r, seed) for r in seed_rows]

# Anything notebook 2 has accepted, added here.
run_dir = Path("data/runs")
if run_dir.exists():
    for f in run_dir.glob("*.jsonl"):
        for line in open(f, encoding="utf8"):
            o = json.loads(line)
            if o.get("status") == "accepted" and o.get("holding"):
                h = o["holding"]
                examples.append(example_for(
                    {"cite": h["cite"], "tradition": h.get("tradition", ""),
                     "situation": h["situation"], "winner": h["winner"],
                     "remedy": h["remedy"], "reason": h["reason"],
                     "text": h.get("restatement", ""), "source": {}}, seed))

print(f"{len(examples)} training examples")
print(f"mean user-turn length: {sum(len(e['messages'][1]['content']) for e in examples)/len(examples):,.0f} chars")

Path("data/train").mkdir(parents=True, exist_ok=True)
with open("data/train/sft.jsonl", "w", encoding="utf8") as fh:
    for e in examples:
        fh.write(json.dumps(e, ensure_ascii=False) + "\n")
print("wrote data/train/sft.jsonl")

## 3. Split

Held out **by tradition**, not at random. A random split lets the model see
Exodus 21:29 while being tested on Exodus 21:36, which is nearly the same
sentence; the resulting number measures memorisation. Holding out a whole
tradition asks the real question: does the factor vocabulary learned on Rome
transfer to Babylon?

In [ ]:
HELDOUT_TRADITION = "mesopotamian"
train = [e for e in examples if e["tradition"] != HELDOUT_TRADITION]
test  = [e for e in examples if e["tradition"] == HELDOUT_TRADITION]
print(f"train {len(train)}  test {len(test)}  (held out: {HELDOUT_TRADITION})")

## 4. Load the base model

Any instruct-tuned model with a chat template. 7-8B in 4-bit fits a T4.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"    # or meta-llama/Llama-3.1-8B-Instruct

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
#
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#                          bnb_4bit_compute_dtype=torch.bfloat16,
#                          bnb_4bit_use_double_quant=True)
# tok = AutoTokenizer.from_pretrained(MODEL_ID)
# tok.pad_token = tok.pad_token or tok.eos_token
# model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
#                                              device_map="auto",
#                                              torch_dtype=torch.bfloat16)
# model.config.use_cache = False
print("model id:", MODEL_ID)

## 5. LoRA + SFT

In [ ]:
# from peft import LoraConfig
# from trl import SFTConfig, SFTTrainer
# from datasets import Dataset
#
# peft_config = LoraConfig(
#     r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj","k_proj","v_proj","o_proj",
#                     "gate_proj","up_proj","down_proj"])
#
# args = SFTConfig(
#     output_dir="checkpoints/nomos-sft",
#     num_train_epochs=3,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=1e-4,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     logging_steps=5,
#     save_strategy="epoch",
#     bf16=True,
#     max_seq_length=6144,     # the precedent block is long; do not truncate it
#     gradient_checkpointing=True,
#     report_to=[],
# )
#
# trainer = SFTTrainer(
#     model=model, args=args, peft_config=peft_config,
#     train_dataset=Dataset.from_list([{"messages": e["messages"]} for e in train]),
#     eval_dataset=Dataset.from_list([{"messages": e["messages"]} for e in test]),
#     processing_class=tok,
# )
# trainer.train()
# trainer.save_model("checkpoints/nomos-sft")
print("training cell -- uncomment to run")

## 6. Expert iteration

The part that makes this worth doing. Sample k candidates per provision at
temperature, verify each with Lean, keep only those that pass all three checks,
and add them to the training set. The verifier is the filter, so the model is
never trained on a formalisation that contradicts the corpus.

Two guards worth keeping:

* **Do not keep a sample merely because it compiles.** Require `well_formed`
  and `coheres`. A compiling-but-incoherent sample teaches the model to
  contradict the corpus fluently.
* **Deduplicate by holding, not by text.** Several samples will converge on the
  same `Holding` with different prose; keeping all of them over-weights easy
  provisions.

In [ ]:
from nomos.verify import verify, lean_available
from nomos.pipeline import parse_json
from nomos.factors import BY_NAME

def expert_iterate(provisions, corpus, generate, k=4, require_coherence=True):
    """`generate(system, user, n) -> list[str]` samples n completions."""
    kept, seen = [], set()
    for p in provisions:
        guess = p.get("situation")
        user = prompts.build_prompt(p, corpus, guess_situation=guess, k=6)
        for raw in generate(prompts.SYSTEM, user, k):
            d = parse_json(raw)
            if not d or d.get("applicable") is False:
                continue
            sit = [f for f in d.get("situation", []) if f in BY_NAME]
            rsn = [f for f in d.get("reason", []) if f in BY_NAME]
            h = Holding(cite=p.get("citation", "?"), situation=sit,
                        winner=("respondent" if d.get("winner") == "respondent"
                                else "claimant"),
                        remedy=d.get("remedy", "Remedy.exempt"), reason=rsn,
                        tradition=p.get("tradition", ""))
            key = (h.cite, tuple(sorted(sit)), h.winner, h.remedy, tuple(sorted(rsn)))
            if key in seen:
                continue
            seen.add(key)
            r = verify(h, h.tradition) if lean_available() else None
            if r and r.ok and (r.coheres is not False or not require_coherence):
                kept.append((h, r))
    return kept

print("expert_iterate defined; supply a `generate` and a provision list to run it")

## 7. Evaluate the fine-tuned model

Same stratified harness as notebook 2, so the numbers are comparable to the
baselines there. Quote the **open** stratum.

In [ ]:
from nomos import eval as ev, baselines
from nomos.pipeline import HFBackend, formalize

# backend = HFBackend("checkpoints/nomos-sft")
# base = [h for h in seed if h.tradition != HELDOUT_TRADITION]
# gold = [h for h in seed if h.tradition == HELDOUT_TRADITION]
# preds = {}
# for h in gold:
#     row = next(r for r in seed_rows if r["cite"] == h.cite)
#     p = {"citation": h.cite, "tradition": h.tradition, "text": row.get("text"),
#          "canonical": h.cite, "work": h.tradition, "language": "en", "source": {}}
#     o = formalize(p, base, backend, max_repairs=1, verify_with_lean=lean_available())
#     if o.holding:
#         preds[h.cite] = o.holding
# rep = ev.evaluate(gold, preds, base)
# print(rep.render())
print("evaluation cell -- uncomment after training")

## What success would look like

Worth writing down before running anything, so the goalposts stay put.

* **Ratio agreement above chance on the open stratum.** Outcome agreement is
  easy — most provisions find for the claimant. Agreeing on *which facts
  mattered* is the thing precedent actually transmits, and `ratio_jaccard` on
  open cases is the number to watch.
* **Transfer across traditions.** Train without Mesopotamia, test on it. If
  that works, the factor vocabulary is capturing something about legal
  structure rather than about the idiom of a particular text.
* **The verifier catching real errors, not just typos.** Track what fraction of
  rejections are well-formedness or coherence failures rather than syntax. A
  model whose only failures are syntax is not being asked hard enough
  questions.
* **A falsifier.** If a model trained on Rome and the Mishnah can formalise
  Hammurabi as accurately as one trained on Hammurabi directly, the factor
  vocabulary is doing real work. If it cannot, the vocabulary is
  tradition-specific and the whole comparative programme needs rethinking.
  That is the experiment this repository exists to make runnable.